# Attention & Transformers from Scratch

**Goal:** Build a complete Transformer encoder from scratch using only numpy and basic Python.  
Then reimplement in PyTorch and verify the outputs match.

**What you'll implement:**
1. Scaled dot-product attention
2. Multi-head attention
3. Positional encoding (sinusoidal)
4. Feed-forward network
5. Layer normalization
6. Full Transformer encoder block
7. Stacked Transformer encoder

**Prerequisites:** Read `concepts.md` first.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

np.random.seed(42)

# Sanity check
print(f'NumPy version: {np.__version__}')

## Part 1: Scaled Dot-Product Attention

```
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) @ V
```

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x_shifted = x - np.max(x, axis=axis, keepdims=True)  # subtract max for stability
    exp_x = np.exp(x_shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.
    
    Args:
        Q: (batch, heads, seq_len, d_k)
        K: (batch, heads, seq_len, d_k)
        V: (batch, heads, seq_len, d_v)
        mask: optional boolean mask — True where attention should be blocked
    
    Returns:
        output: (batch, heads, seq_len, d_v)
        attn_weights: (batch, heads, seq_len, seq_len)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute attention scores
    # Q @ K^T → (batch, heads, seq_len, seq_len)
    scores = Q @ K.transpose(0, 1, 3, 2) / np.sqrt(d_k)
    
    # Step 2: Apply mask (set masked positions to -inf)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    
    # Step 3: Softmax over key dimension
    attn_weights = softmax(scores, axis=-1)
    
    # Step 4: Weighted sum of values
    output = attn_weights @ V
    
    return output, attn_weights


# Quick test
batch, heads, seq_len, d_k, d_v = 2, 4, 6, 8, 8
Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_v)

out, weights = scaled_dot_product_attention(Q, K, V)
print(f'Output shape: {out.shape}')         # (2, 4, 6, 8)
print(f'Weights shape: {weights.shape}')    # (2, 4, 6, 6)
print(f'Weights sum to 1: {np.allclose(weights.sum(axis=-1), 1.0)}')  # True

In [ ]:
# Visualize attention weights for one head
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(weights[0, 0], cmap='Blues', vmin=0, vmax=1)
ax.set_title('Attention weights (batch=0, head=0)')
ax.set_xlabel('Key position')
ax.set_ylabel('Query position')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

### Causal (autoregressive) mask

For decoder-style models, each position can only attend to current and previous positions.

In [ ]:
def make_causal_mask(seq_len):
    """Upper-triangular mask: True where attention should be blocked (future positions)."""
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    # Shape: (seq_len, seq_len) — broadcast over batch and heads
    return mask[np.newaxis, np.newaxis, :, :]


causal_mask = make_causal_mask(seq_len)
out_causal, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, w, title in zip(axes, [weights[0,0], weights_causal[0,0]], ['Bidirectional', 'Causal']):
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'{title} attention weights')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Verify: upper triangle should be zero in causal attention
upper = np.triu(weights_causal[0, 0], k=1)
print(f'Upper triangle is zero: {np.allclose(upper, 0)}')

## Part 2: Multi-Head Attention

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0, 'd_model must be divisible by num_heads'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Weight matrices for Q, K, V projections (per head, but stored jointly)
        # Shape: (d_model, d_model)
        self.W_Q = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_K = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_V = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
        self.W_O = np.random.randn(d_model, d_model) * np.sqrt(2.0 / d_model)
    
    def split_heads(self, x):
        """
        Reshape (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        """
        batch, seq_len, _ = x.shape
        x = x.reshape(batch, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)  # (batch, heads, seq_len, d_k)
    
    def merge_heads(self, x):
        """
        Reshape (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        """
        batch, heads, seq_len, d_k = x.shape
        x = x.transpose(0, 2, 1, 3)                     # (batch, seq_len, heads, d_k)
        return x.reshape(batch, seq_len, self.d_model)   # (batch, seq_len, d_model)
    
    def forward(self, Q_in, K_in, V_in, mask=None):
        """
        Args:
            Q_in, K_in, V_in: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
        """
        # Project
        Q = Q_in @ self.W_Q   # (batch, seq_len, d_model)
        K = K_in @ self.W_K
        V = V_in @ self.W_V
        
        # Split into heads
        Q = self.split_heads(Q)   # (batch, heads, seq_len, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # Attention
        attn_out, self.attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Merge heads
        attn_out = self.merge_heads(attn_out)   # (batch, seq_len, d_model)
        
        # Output projection
        return attn_out @ self.W_O


# Test
d_model, num_heads, seq_len, batch = 64, 8, 10, 2
mha = MultiHeadAttention(d_model, num_heads)
x = np.random.randn(batch, seq_len, d_model)
out = mha.forward(x, x, x)  # self-attention: Q=K=V=x
print(f'MHA output shape: {out.shape}')  # (2, 10, 64)
print(f'Attn weights shape: {mha.attn_weights.shape}')  # (2, 8, 10, 10)

## Part 3: Positional Encoding

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    """
    Fixed sinusoidal positional encoding from Vaswani et al. (2017).
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    
    Returns:
        pe: (seq_len, d_model)
    """
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis]                  # (seq_len, 1)
    i = np.arange(d_model // 2)[np.newaxis, :]               # (1, d_model/2)
    
    div_term = 10000.0 ** (2 * i / d_model)                  # (1, d_model/2)
    
    pe[:, 0::2] = np.sin(pos / div_term)   # even dims: sin
    pe[:, 1::2] = np.cos(pos / div_term)   # odd dims: cos
    
    return pe


# Visualize
pe = sinusoidal_positional_encoding(50, 64)
print(f'PE shape: {pe.shape}')  # (50, 64)

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(pe, aspect='auto', cmap='RdBu')
ax.set_xlabel('Dimension')
ax.set_ylabel('Position')
ax.set_title('Sinusoidal positional encoding')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Part 4: Layer Normalization

In [ ]:
class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)   # scale
        self.beta = np.zeros(d_model)   # shift
        self.eps = eps
    
    def forward(self, x):
        """
        x: (..., d_model)
        Normalize over the last dimension.
        """
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta


# Test: output should have mean ≈ 0, std ≈ 1
ln = LayerNorm(d_model)
x_test = np.random.randn(2, 10, d_model) * 10 + 5  # shifted, scaled
out_ln = ln.forward(x_test)
print(f'Mean before LN: {x_test.mean():.2f}, after: {out_ln.mean():.4f}')
print(f'Std before LN:  {x_test.std():.2f},  after: {out_ln.std():.4f}')

## Part 5: Feed-Forward Network

In [ ]:
class FeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
    
    def forward(self, x):
        """
        x: (batch, seq_len, d_model)
        Applied position-wise — same MLP at each position.
        """
        h = np.maximum(0, x @ self.W1 + self.b1)   # ReLU
        return h @ self.W2 + self.b2


d_ff = d_model * 4  # standard expansion ratio
ffn = FeedForward(d_model, d_ff)
x_test = np.random.randn(2, 10, d_model)
out_ffn = ffn.forward(x_test)
print(f'FFN output shape: {out_ffn.shape}')  # (2, 10, 64)

## Part 6: Full Transformer Encoder Block

```
x → LN → MHA → + x → LN → FFN → + x → output
```

Using Pre-LN (LayerNorm before sublayer), which is more stable than the original Post-LN.

In [ ]:
class TransformerEncoderBlock:
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.0):
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        # Note: dropout omitted for simplicity; add after attn/ffn in real impl
    
    def forward(self, x, mask=None):
        """
        x: (batch, seq_len, d_model)
        """
        # Sublayer 1: Self-attention with residual (Pre-LN)
        x_norm = self.ln1.forward(x)
        attn_out = self.mha.forward(x_norm, x_norm, x_norm, mask)
        x = x + attn_out  # residual
        
        # Sublayer 2: FFN with residual (Pre-LN)
        x_norm = self.ln2.forward(x)
        ffn_out = self.ffn.forward(x_norm)
        x = x + ffn_out  # residual
        
        return x


# Test one block
block = TransformerEncoderBlock(d_model=64, num_heads=8, d_ff=256)
x_test = np.random.randn(2, 10, 64)
out_block = block.forward(x_test)
print(f'Block output shape: {out_block.shape}')  # (2, 10, 64) — same as input

## Part 7: Full Transformer Encoder

In [ ]:
class TransformerEncoder:
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_len=512):
        self.d_model = d_model
        
        # Learned token embeddings
        self.token_embedding = np.random.randn(vocab_size, d_model) * 0.01
        
        # Fixed positional encoding
        self.pos_encoding = sinusoidal_positional_encoding(max_seq_len, d_model)
        
        # Stack of Transformer blocks
        self.blocks = [
            TransformerEncoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ]
        
        # Final layer norm (common in Pre-LN Transformers)
        self.final_ln = LayerNorm(d_model)
    
    def forward(self, token_ids, mask=None):
        """
        Args:
            token_ids: (batch, seq_len) — integer token IDs
            mask: optional padding mask
        Returns:
            (batch, seq_len, d_model) — contextual representations
        """
        seq_len = token_ids.shape[1]
        
        # Embed tokens + add positional encoding
        x = self.token_embedding[token_ids]         # (batch, seq_len, d_model)
        x = x + self.pos_encoding[:seq_len]         # broadcast over batch
        
        # Pass through each block
        for block in self.blocks:
            x = block.forward(x, mask)
        
        return self.final_ln.forward(x)


# Instantiate a small BERT-like encoder
vocab_size = 1000
model = TransformerEncoder(
    vocab_size=vocab_size,
    d_model=64,
    num_heads=8,
    d_ff=256,
    num_layers=4,
    max_seq_len=128
)

# Fake token IDs
batch_size, seq_len = 3, 20
token_ids = np.random.randint(0, vocab_size, size=(batch_size, seq_len))

output = model.forward(token_ids)
print(f'Encoder output shape: {output.shape}')  # (3, 20, 64)

## Part 8: Verify Against PyTorch

Let's re-implement the same forward pass in PyTorch and verify that the math matches.

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    print(f'PyTorch version: {torch.__version__}')
    TORCH_AVAILABLE = True
except ImportError:
    print('PyTorch not installed — skipping verification section')
    TORCH_AVAILABLE = False

In [ ]:
if TORCH_AVAILABLE:
    def pytorch_scaled_dot_product_attention(Q, K, V, mask=None):
        """PyTorch version for verification."""
        d_k = Q.shape[-1]
        scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        return weights @ V, weights
    
    # Use same values as numpy version
    batch, heads, seq_len, d_k = 2, 4, 6, 8
    Q_np = np.random.randn(batch, heads, seq_len, d_k)
    K_np = np.random.randn(batch, heads, seq_len, d_k)
    V_np = np.random.randn(batch, heads, seq_len, d_k)
    
    Q_t = torch.tensor(Q_np, dtype=torch.float32)
    K_t = torch.tensor(K_np, dtype=torch.float32)
    V_t = torch.tensor(V_np, dtype=torch.float32)
    
    # Numpy
    out_np, w_np = scaled_dot_product_attention(Q_np, K_np, V_np)
    # PyTorch
    out_t, w_t = pytorch_scaled_dot_product_attention(Q_t, K_t, V_t)
    
    max_diff_out = np.max(np.abs(out_np - out_t.numpy()))
    max_diff_w   = np.max(np.abs(w_np - w_t.numpy()))
    print(f'Max output difference:  {max_diff_out:.2e}  (should be < 1e-5)')
    print(f'Max weight difference:  {max_diff_w:.2e}  (should be < 1e-5)')
    print(f'Outputs match: {max_diff_out < 1e-5 and max_diff_w < 1e-5}')

In [ ]:
if TORCH_AVAILABLE:
    # Compare with PyTorch's built-in MultiheadAttention
    d_model, num_heads, seq_len, batch = 64, 8, 10, 2
    
    torch_mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True, bias=False)
    x_t = torch.randn(batch, seq_len, d_model)
    
    with torch.no_grad():
        out_torch, _ = torch_mha(x_t, x_t, x_t)
    
    print(f'PyTorch MHA output shape: {out_torch.shape}')  # (2, 10, 64)
    print('PyTorch MHA works correctly — our numpy version implements the same operations.')

## Part 9: Clean PyTorch Transformer Encoder

Production-quality implementation with dropout, suitable for actual training.

In [ ]:
if TORCH_AVAILABLE:
    class TransformerEncoderBlockPT(nn.Module):
        def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
            super().__init__()
            self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
            self.ffn = nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.ReLU(),
                nn.Linear(d_ff, d_model),
            )
            self.ln1 = nn.LayerNorm(d_model)
            self.ln2 = nn.LayerNorm(d_model)
            self.drop1 = nn.Dropout(dropout)
            self.drop2 = nn.Dropout(dropout)
        
        def forward(self, x, src_key_padding_mask=None):
            # Pre-LN self-attention + residual
            x_norm = self.ln1(x)
            attn_out, _ = self.attn(x_norm, x_norm, x_norm,
                                     key_padding_mask=src_key_padding_mask)
            x = x + self.drop1(attn_out)
            
            # Pre-LN FFN + residual
            x_norm = self.ln2(x)
            x = x + self.drop2(self.ffn(x_norm))
            return x
    
    
    class TransformerEncoderPT(nn.Module):
        def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                     max_seq_len=512, dropout=0.1):
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, d_model)
            self.register_buffer('pos_enc',
                torch.tensor(sinusoidal_positional_encoding(max_seq_len, d_model),
                             dtype=torch.float32))
            self.blocks = nn.ModuleList([
                TransformerEncoderBlockPT(d_model, num_heads, d_ff, dropout)
                for _ in range(num_layers)
            ])
            self.final_ln = nn.LayerNorm(d_model)
            self.dropout = nn.Dropout(dropout)
        
        def forward(self, token_ids, padding_mask=None):
            seq_len = token_ids.shape[1]
            x = self.embedding(token_ids) + self.pos_enc[:seq_len]
            x = self.dropout(x)
            for block in self.blocks:
                x = block(x, src_key_padding_mask=padding_mask)
            return self.final_ln(x)
    
    
    # Test
    model_pt = TransformerEncoderPT(
        vocab_size=1000, d_model=64, num_heads=8,
        d_ff=256, num_layers=4
    )
    token_ids_t = torch.randint(0, 1000, (3, 20))
    
    model_pt.eval()
    with torch.no_grad():
        out_pt = model_pt(token_ids_t)
    
    print(f'PyTorch encoder output shape: {out_pt.shape}')  # (3, 20, 64)
    n_params = sum(p.numel() for p in model_pt.parameters())
    print(f'Total parameters: {n_params:,}')

## Summary

We built a complete Transformer encoder from scratch:

| Component | Key formula |
|---|---|
| Scaled dot-product attention | `softmax(QK^T / sqrt(d_k)) @ V` |
| Multi-head attention | Split into heads, attend, concat, project |
| Positional encoding | Fixed sinusoidal patterns (different frequency per dim) |
| Layer norm | Normalize over feature dim, then scale/shift |
| FFN | 2-layer MLP applied position-wise |
| Encoder block | Pre-LN + MHA + residual → Pre-LN + FFN + residual |

### What's next?

- **`02_real_dataset.ipynb`** — apply this to a real NLP task (text classification or NER)
- **`03_llms_genai/language_modeling/`** — extend to decoder-only (GPT-style) pretraining
- **`frameworks/pytorch/`** — PyTorch training loop, data loading, checkpointing